# 06 — Empirical Analysis

This notebook tests whether Bitcoin prediction market probabilities systematically differ from the Deribit-implied probability proxy.

The empirical analysis focuses on mean mispricing tests, confidence intervals, regression models, and robustness checks across the main Polymarket terminal sample, supplementary Kalshi sample, and alternative Polymarket robustness samples.

In [2]:
# ============================================================
# Imports and project configuration
# ============================================================

import numpy as np
import pandas as pd

from scipy import stats
import statsmodels.formula.api as smf

from importlib import reload
import config
reload(config)

from config import *

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Final data directory:", FINAL_DIR)
print("Tables directory:", TABLES_DIR)

Project directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project
Processed data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
Final data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final
Tables directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables


In [3]:
# ============================================================
# Load empirical datasets
# ============================================================

poly_terminal_path = (
    PROCESSED_DIR / "polymarket_deribit_matched_terminal.csv"
)
poly_terminal_short_path = (
    PROCESSED_DIR / "polymarket_deribit_matched_terminal_short.csv"
)
poly_terminal_very_short_path = (
    PROCESSED_DIR
    / "polymarket_deribit_matched_terminal_very_short.csv"
)
poly_path_dep_path = (
    PROCESSED_DIR
    / "polymarket_deribit_matched_path_dependent.csv"
)

kalshi_matched_path = (
    PROCESSED_DIR / "kalshi_deribit_matched.csv"
)
kalshi_matched_15m_path = (
    PROCESSED_DIR / "kalshi_deribit_matched_15m.csv"
)

unified_path = (
    FINAL_DIR
    / "unified_prediction_market_deribit_matched.csv"
)

df_poly_terminal = pd.read_csv(poly_terminal_path)
df_poly_terminal_short = pd.read_csv(poly_terminal_short_path)
df_poly_terminal_very_short = pd.read_csv(
    poly_terminal_very_short_path
)
df_poly_path_dep = pd.read_csv(poly_path_dep_path)

df_kalshi = pd.read_csv(kalshi_matched_path)
df_kalshi_15m = pd.read_csv(kalshi_matched_15m_path)
df_unified = pd.read_csv(unified_path)

for df in [
    df_poly_terminal,
    df_poly_terminal_short,
    df_poly_terminal_very_short,
    df_poly_path_dep,
]:
    for col in ["date", "datetime", "end_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                utc=True,
                errors="coerce",
            )

for col in [
    "date",
    "observation_time",
    "expiration_time",
]:
    if col in df_unified.columns:
        df_unified[col] = pd.to_datetime(
            df_unified[col],
            utc=True,
            errors="coerce",
        )

for df in [df_kalshi, df_kalshi_15m]:
    for col in [
        "observation_date",
        "observation_time",
        "open_time",
        "close_time",
        "benchmark_available_at",
        "spot_available_at",
    ]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                utc=True,
                errors="coerce",
            )

# Kalshi validation.
assert len(df_kalshi) == 3_135
assert len(df_kalshi_15m) == 2_554

assert df_kalshi["ticker"].is_unique
assert df_kalshi_15m["ticker"].is_unique

assert df_kalshi["price_source"].eq(
    "first_timestamped_trade"
).all()
assert df_kalshi_15m["price_source"].eq(
    "first_timestamped_trade"
).all()

assert df_kalshi["minutes_after_open"].between(
    0,
    30,
    inclusive="both",
).all()

assert df_kalshi_15m["minutes_after_open"].between(
    0,
    15,
    inclusive="both",
).all()

assert set(df_kalshi_15m["ticker"]).issubset(
    set(df_kalshi["ticker"])
)

if "benchmark_available_at" in df_kalshi.columns:
    assert (
        df_kalshi["benchmark_available_at"]
        <= df_kalshi["observation_time"]
    ).all()

if "spot_available_at" in df_kalshi.columns:
    assert (
        df_kalshi["spot_available_at"]
        <= df_kalshi["observation_time"]
    ).all()

required_cols = [
    "mispricing",
    "prob_market",
    "prob_deribit",
    "tau",
    "sigma",
]

for sample_name, df_sample in {
    "Polymarket terminal": df_poly_terminal,
    "Polymarket path-dependent": df_poly_path_dep,
    "Kalshi 30m": df_kalshi,
    "Kalshi 15m": df_kalshi_15m,
    "Unified": df_unified,
}.items():
    missing = [
        col for col in required_cols
        if col not in df_sample.columns
    ]
    if missing:
        raise ValueError(
            f"{sample_name} missing columns: {missing}"
        )

print("Empirical input validation passed.")

print("=" * 70)
print("Loaded empirical analysis datasets")
print("=" * 70)

print("Polymarket terminal:")
print(f"  Rows: {len(df_poly_terminal):,}")
print(
    f"  Markets: "
    f"{df_poly_terminal['market_id'].nunique():,}"
)

print("\nPolymarket terminal short:")
print(f"  Rows: {len(df_poly_terminal_short):,}")

print("\nPolymarket terminal very short:")
print(f"  Rows: {len(df_poly_terminal_very_short):,}")

print("\nPolymarket path-dependent:")
print(f"  Rows: {len(df_poly_path_dep):,}")
print(
    f"  Markets: "
    f"{df_poly_path_dep['market_id'].nunique():,}"
)

print("\nKalshi first trade within 30 minutes:")
print(f"  Rows: {len(df_kalshi):,}")
print(
    f"  Events: "
    f"{df_kalshi['event_ticker'].nunique():,}"
)

print("\nKalshi first trade within 15 minutes:")
print(f"  Rows: {len(df_kalshi_15m):,}")
print(
    f"  Events: "
    f"{df_kalshi_15m['event_ticker'].nunique():,}"
)

print("\nUnified:")
print(f"  Rows: {len(df_unified):,}")
print(
    f"  Contracts: "
    f"{df_unified['contract_id'].nunique():,}"
)

Empirical input validation passed.
Loaded empirical analysis datasets
Polymarket terminal:
  Rows: 19,649
  Markets: 2,847

Polymarket terminal short:
  Rows: 19,649

Polymarket terminal very short:
  Rows: 19,616

Polymarket path-dependent:
  Rows: 10,323
  Markets: 584

Kalshi first trade within 30 minutes:
  Rows: 3,135
  Events: 485

Kalshi first trade within 15 minutes:
  Rows: 2,554
  Events: 483

Unified:
  Rows: 33,107
  Contracts: 6,566


In [4]:
# ============================================================
# Define empirical samples and analysis variables
# ============================================================

KALSHI_MAIN_LABEL = (
    "Kalshi timestamped, first trade within 30 minutes"
)
KALSHI_15M_LABEL = (
    "Kalshi timestamped, first trade within 15 minutes"
)

if "event_id_unified" not in df_unified.columns:
    raise ValueError(
        "Unified dataset is missing event_id_unified."
    )

df_unified["inference_cluster"] = np.where(
    df_unified["market"].eq("kalshi"),
    (
        "kalshi-event:"
        + df_unified["event_id_unified"].astype(str)
    ),
    (
        "polymarket-market:"
        + df_unified["contract_id"].astype(str)
    ),
)

empirical_samples = {
    "Polymarket terminal":
        df_poly_terminal.copy(),
    "Polymarket terminal short-horizon":
        df_poly_terminal_short.copy(),
    "Polymarket terminal very-short-horizon":
        df_poly_terminal_very_short.copy(),
    "Polymarket path-dependent":
        df_poly_path_dep.copy(),
    KALSHI_MAIN_LABEL:
        df_kalshi.copy(),
    KALSHI_15M_LABEL:
        df_kalshi_15m.copy(),
    "Unified matched":
        df_unified.copy(),
}

for sample_name, df_sample in empirical_samples.items():
    if (df_sample["tau"] <= 0).any():
        raise ValueError(
            f"{sample_name} contains non-positive tau."
        )

    df_sample["abs_mispricing"] = (
        df_sample["mispricing"].abs()
    )
    df_sample["squared_mispricing"] = (
        df_sample["mispricing"] ** 2
    )
    df_sample["tau_days"] = (
        df_sample["tau"] * 365
    )
    df_sample["tau_hours"] = (
        df_sample["tau"] * 365 * 24
    )
    df_sample["log_tau"] = np.log(
        df_sample["tau"]
    )

    empirical_samples[sample_name] = df_sample

df_poly_terminal = empirical_samples[
    "Polymarket terminal"
]
df_poly_terminal_short = empirical_samples[
    "Polymarket terminal short-horizon"
]
df_poly_terminal_very_short = empirical_samples[
    "Polymarket terminal very-short-horizon"
]
df_poly_path_dep = empirical_samples[
    "Polymarket path-dependent"
]
df_kalshi = empirical_samples[KALSHI_MAIN_LABEL]
df_kalshi_15m = empirical_samples[KALSHI_15M_LABEL]
df_unified = empirical_samples["Unified matched"]

print("=" * 70)
print("Empirical samples defined")
print("=" * 70)

for sample_name, df_sample in empirical_samples.items():
    if "market_id" in df_sample.columns:
        identifier = "market_id"
    elif "ticker" in df_sample.columns:
        identifier = "ticker"
    else:
        identifier = "contract_id"

    print(sample_name)
    print(f"  Rows: {len(df_sample):,}")
    print(
        f"  Contracts: "
        f"{df_sample[identifier].nunique():,}"
    )
    print(
        f"  Mean mispricing: "
        f"{df_sample['mispricing'].mean():.6f}"
    )
    print(
        f"  Mean tau days: "
        f"{df_sample['tau_days'].mean():.4f}"
    )

Empirical samples defined
Polymarket terminal
  Rows: 19,649
  Contracts: 2,847
  Mean mispricing: 0.007551
  Mean tau days: 3.6941
Polymarket terminal short-horizon
  Rows: 19,649
  Contracts: 2,847
  Mean mispricing: 0.007551
  Mean tau days: 3.6941
Polymarket terminal very-short-horizon
  Rows: 19,616
  Contracts: 2,847
  Mean mispricing: 0.007511
  Mean tau days: 3.6780
Polymarket path-dependent
  Rows: 10,323
  Contracts: 584
  Mean mispricing: 0.084643
  Mean tau days: 54.9478
Kalshi timestamped, first trade within 30 minutes
  Rows: 3,135
  Contracts: 3,135
  Mean mispricing: 0.063073
  Mean tau days: 0.0363
Kalshi timestamped, first trade within 15 minutes
  Rows: 2,554
  Contracts: 2,554
  Mean mispricing: 0.063855
  Mean tau days: 0.0384
Unified matched
  Rows: 33,107
  Contracts: 6,566
  Mean mispricing: 0.036846
  Mean tau days: 19.3290


In [5]:
# ============================================================
# Helper function: mean mispricing test
# ============================================================

def infer_contract_column(df):
    if "market_id" in df.columns:
        return "market_id"
    if "ticker" in df.columns:
        return "ticker"
    if "contract_id" in df.columns:
        return "contract_id"
    raise ValueError(
        "No contract identifier column found."
    )


def mean_mispricing_test(
    df,
    sample_name,
    value_col="mispricing",
):
    x = df[value_col].dropna().astype(float)
    n = len(x)

    if n < 2:
        raise ValueError(
            f"{sample_name} has fewer than two observations."
        )

    identifier = infer_contract_column(df)
    n_contracts = (
        df.loc[x.index, identifier].nunique()
    )

    mean_value = x.mean()
    median_value = x.median()
    sd_value = x.std(ddof=1)
    se_value = sd_value / np.sqrt(n)

    if se_value == 0:
        raise ValueError(
            f"{sample_name} has zero standard error."
        )

    t_stat = mean_value / se_value
    p_value = 2 * stats.t.sf(
        abs(t_stat),
        df=n - 1,
    )

    critical_value = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    return {
        "sample": sample_name,
        "observations": n,
        "contracts": n_contracts,
        "mean_mispricing": mean_value,
        "median_mispricing": median_value,
        "sd_mispricing": sd_value,
        "std_error": se_value,
        "t_stat": t_stat,
        "p_value": p_value,
        "ci_95_low": (
            mean_value
            - critical_value * se_value
        ),
        "ci_95_high": (
            mean_value
            + critical_value * se_value
        ),
    }

In [6]:
# ============================================================
# Mean mispricing tests by empirical sample
# ============================================================

mean_test_rows = [
    mean_mispricing_test(
        df_sample,
        sample_name,
    )
    for sample_name, df_sample
    in empirical_samples.items()
]

mean_mispricing_tests = pd.DataFrame(
    mean_test_rows
)

round_cols = [
    "mean_mispricing",
    "median_mispricing",
    "sd_mispricing",
    "std_error",
    "t_stat",
    "p_value",
    "ci_95_low",
    "ci_95_high",
]

mean_mispricing_tests[round_cols] = (
    mean_mispricing_tests[round_cols].round(6)
)

mean_tests_path = (
    TABLES_DIR / "mean_mispricing_tests.csv"
)
mean_mispricing_tests.to_csv(
    mean_tests_path,
    index=False,
)

display(mean_mispricing_tests)
print(
    f"Saved mean mispricing tests to: "
    f"{mean_tests_path}"
)

,sample,observations,contracts,mean_mispricing,median_mispricing,sd_mispricing,std_error,t_stat,p_value,ci_95_low,ci_95_high
0,Polymarket terminal,19649,2847,0.007551,0.002772,0.039818,0.000284,26.580929,0.0,0.006994,0.008107
1,Polymarket terminal short-horizon,19649,2847,0.007551,0.002772,0.039818,0.000284,26.580929,0.0,0.006994,0.008107
2,Polymarket terminal very-short-horizon,19616,2847,0.007511,0.002740,0.039828,0.000284,26.413494,0.0,0.006954,0.008069
3,Polymarket path-dependent,10323,584,0.084643,0.030254,0.143066,0.001408,60.111877,0.0,0.081883,0.087404
4,"Kalshi timestamped, first trade within 30 minutes",3135,3135,0.063073,0.019875,0.147670,0.002637,23.915031,0.0,0.057902,0.068244
5,"Kalshi timestamped, first trade within 15 minutes",2554,2554,0.063855,0.026789,0.139220,0.002755,23.179580,0.0,0.058453,0.069257
6,Unified matched,33107,6566,0.036846,0.008653,0.103314,0.000568,64.892538,0.0,0.035733,0.037959


Saved mean mispricing tests to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/mean_mispricing_tests.csv


In [7]:
# ============================================================
# Row-level bootstrap confidence intervals
# ============================================================

def row_bootstrap_ci(
    df,
    sample_name,
    value_col="mispricing",
    n_boot=5000,
    random_seed=42,
):
    x = (
        df[value_col]
        .dropna()
        .astype(float)
        .to_numpy()
    )
    n = len(x)

    if n < 2:
        raise ValueError(
            f"{sample_name} has fewer than two observations."
        )

    rng = np.random.default_rng(random_seed)

    boot_means = np.empty(n_boot)
    boot_medians = np.empty(n_boot)

    for b in range(n_boot):
        sample = rng.choice(
            x,
            size=n,
            replace=True,
        )
        boot_means[b] = sample.mean()
        boot_medians[b] = np.median(sample)

    return {
        "sample": sample_name,
        "observations": n,
        "mean_mispricing": x.mean(),
        "mean_ci_95_low": np.percentile(
            boot_means,
            2.5,
        ),
        "mean_ci_95_high": np.percentile(
            boot_means,
            97.5,
        ),
        "median_mispricing": np.median(x),
        "median_ci_95_low": np.percentile(
            boot_medians,
            2.5,
        ),
        "median_ci_95_high": np.percentile(
            boot_medians,
            97.5,
        ),
        "n_boot": n_boot,
    }


row_bootstrap_rows = [
    row_bootstrap_ci(
        df_sample,
        sample_name,
    )
    for sample_name, df_sample
    in empirical_samples.items()
]

row_bootstrap_results = pd.DataFrame(
    row_bootstrap_rows
)

row_bootstrap_round_cols = [
    "mean_mispricing",
    "mean_ci_95_low",
    "mean_ci_95_high",
    "median_mispricing",
    "median_ci_95_low",
    "median_ci_95_high",
]

row_bootstrap_results[
    row_bootstrap_round_cols
] = row_bootstrap_results[
    row_bootstrap_round_cols
].round(6)

row_bootstrap_path = (
    TABLES_DIR
    / "row_level_bootstrap_mispricing_ci.csv"
)
row_bootstrap_results.to_csv(
    row_bootstrap_path,
    index=False,
)

display(row_bootstrap_results)
print(
    "Saved row-level bootstrap results to: "
    f"{row_bootstrap_path}"
)

,sample,observations,mean_mispricing,mean_ci_95_low,mean_ci_95_high,median_mispricing,median_ci_95_low,median_ci_95_high,n_boot
0,Polymarket terminal,19649,0.007551,0.007005,0.008118,0.002772,0.002500,0.002999,5000
1,Polymarket terminal short-horizon,19649,0.007551,0.007005,0.008118,0.002772,0.002500,0.002999,5000
2,Polymarket terminal very-short-horizon,19616,0.007511,0.006944,0.008076,0.002740,0.002499,0.002993,5000
3,Polymarket path-dependent,10323,0.084643,0.081919,0.087374,0.030254,0.028859,0.031930,5000
4,"Kalshi timestamped, first trade within 30 minutes",3135,0.063073,0.057992,0.068267,0.019875,0.017340,0.024038,5000
5,"Kalshi timestamped, first trade within 15 minutes",2554,0.063855,0.058467,0.069391,0.026789,0.021475,0.030000,5000
6,Unified matched,33107,0.036846,0.035737,0.038003,0.008653,0.008367,0.009001,5000


Saved row-level bootstrap results to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/row_level_bootstrap_mispricing_ci.csv


In [8]:
# ============================================================
# Cluster-level bootstrap confidence intervals
# ============================================================

def cluster_level_bootstrap_ci(
    df,
    sample_name,
    cluster_col,
    value_col="mispricing",
    n_boot=2000,
    random_seed=42,
):
    cluster_stats = (
        df.groupby(cluster_col)[value_col]
        .agg(["mean", "median"])
        .dropna()
    )

    cluster_means = (
        cluster_stats["mean"]
        .astype(float)
        .to_numpy()
    )
    cluster_medians = (
        cluster_stats["median"]
        .astype(float)
        .to_numpy()
    )

    n_clusters = len(cluster_stats)

    if n_clusters < 2:
        raise ValueError(
            f"{sample_name} has fewer than two clusters."
        )

    rng = np.random.default_rng(random_seed)

    boot_means = np.empty(n_boot)
    boot_medians = np.empty(n_boot)

    for b in range(n_boot):
        sampled_idx = rng.integers(
            0,
            n_clusters,
            size=n_clusters,
        )
        boot_means[b] = (
            cluster_means[sampled_idx].mean()
        )
        boot_medians[b] = np.median(
            cluster_medians[sampled_idx]
        )

    x = df[value_col].dropna().astype(float)

    return {
        "sample": sample_name,
        "observations": len(x),
        "clusters": n_clusters,
        "cluster_variable": cluster_col,
        "row_mean_mispricing": x.mean(),
        "cluster_mean_mispricing": (
            cluster_means.mean()
        ),
        "cluster_mean_ci_95_low": np.percentile(
            boot_means,
            2.5,
        ),
        "cluster_mean_ci_95_high": np.percentile(
            boot_means,
            97.5,
        ),
        "row_median_mispricing": x.median(),
        "cluster_median_mispricing": np.median(
            cluster_medians
        ),
        "cluster_median_ci_95_low": np.percentile(
            boot_medians,
            2.5,
        ),
        "cluster_median_ci_95_high": np.percentile(
            boot_medians,
            97.5,
        ),
        "n_boot": n_boot,
    }


cluster_bootstrap_specs = [
    (
        "Polymarket terminal",
        df_poly_terminal,
        "market_id",
    ),
    (
        "Polymarket terminal short-horizon",
        df_poly_terminal_short,
        "market_id",
    ),
    (
        "Polymarket terminal very-short-horizon",
        df_poly_terminal_very_short,
        "market_id",
    ),
    (
        "Polymarket path-dependent",
        df_poly_path_dep,
        "market_id",
    ),
    (
        KALSHI_MAIN_LABEL,
        df_kalshi,
        "event_ticker",
    ),
    (
        KALSHI_15M_LABEL,
        df_kalshi_15m,
        "event_ticker",
    ),
    (
        "Unified matched",
        df_unified,
        "inference_cluster",
    ),
]

cluster_bootstrap_rows = [
    cluster_level_bootstrap_ci(
        df_sample,
        sample_name,
        cluster_col,
    )
    for sample_name, df_sample, cluster_col
    in cluster_bootstrap_specs
]

cluster_bootstrap_results = pd.DataFrame(
    cluster_bootstrap_rows
)

cluster_bootstrap_round_cols = [
    "row_mean_mispricing",
    "cluster_mean_mispricing",
    "cluster_mean_ci_95_low",
    "cluster_mean_ci_95_high",
    "row_median_mispricing",
    "cluster_median_mispricing",
    "cluster_median_ci_95_low",
    "cluster_median_ci_95_high",
]

cluster_bootstrap_results[
    cluster_bootstrap_round_cols
] = cluster_bootstrap_results[
    cluster_bootstrap_round_cols
].round(6)

cluster_bootstrap_path = (
    TABLES_DIR
    / "cluster_level_bootstrap_mispricing_ci.csv"
)
cluster_bootstrap_results.to_csv(
    cluster_bootstrap_path,
    index=False,
)

display(cluster_bootstrap_results)
print(
    "Saved cluster-level bootstrap results to: "
    f"{cluster_bootstrap_path}"
)

,sample,observations,clusters,cluster_variable,row_mean_mispricing,cluster_mean_mispricing,cluster_mean_ci_95_low,cluster_mean_ci_95_high,row_median_mispricing,cluster_median_mispricing,cluster_median_ci_95_low,cluster_median_ci_95_high,n_boot
0,Polymarket terminal,19649,2847,market_id,0.007551,0.007496,0.006709,0.008285,0.002772,0.002764,0.002407,0.003131,2000
1,Polymarket terminal short-horizon,19649,2847,market_id,0.007551,0.007496,0.006709,0.008285,0.002772,0.002764,0.002407,0.003131,2000
2,Polymarket terminal very-short-horizon,19616,2847,market_id,0.007511,0.007483,0.006704,0.008271,0.002740,0.002764,0.002407,0.003131,2000
3,Polymarket path-dependent,10323,584,market_id,0.084643,0.076566,0.067337,0.085983,0.030254,0.018768,0.015237,0.022116,2000
4,"Kalshi timestamped, first trade within 30 minutes",3135,485,event_ticker,0.063073,0.070670,0.064565,0.077081,0.019875,0.032225,0.026905,0.037922,2000
5,"Kalshi timestamped, first trade within 15 minutes",2554,483,event_ticker,0.063855,0.076731,0.070557,0.083126,0.026789,0.039138,0.033755,0.048955,2000
6,Unified matched,33107,3916,inference_cluster,0.036846,0.025620,0.023693,0.027604,0.008653,0.005984,0.005437,0.006565,2000


Saved cluster-level bootstrap results to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/cluster_level_bootstrap_mispricing_ci.csv


In [9]:
# ============================================================
# Helper function: regression results table
# ============================================================

def regression_to_table(model, model_name):
    params = model.params
    conf_int = model.conf_int()

    rows = []

    for variable in params.index:
        rows.append({
            "model": model_name,
            "variable": variable,
            "coef": params[variable],
            "std_error": model.bse[variable],
            "t_stat": model.tvalues[variable],
            "p_value": model.pvalues[variable],
            "ci_95_low": conf_int.loc[variable, 0],
            "ci_95_high": conf_int.loc[variable, 1],
            "nobs": int(model.nobs),
            "r_squared": model.rsquared,
        })

    return pd.DataFrame(rows)


regression_round_cols = [
    "coef",
    "std_error",
    "t_stat",
    "p_value",
    "ci_95_low",
    "ci_95_high",
    "r_squared",
]

In [10]:
# ============================================================
# Polymarket terminal regressions
# ============================================================

df_reg_poly = df_poly_terminal.copy()

df_reg_poly["bet_type"] = (
    df_reg_poly["bet_type"].astype("category")
)

poly_model_1 = smf.ols(
    "mispricing ~ tau_days + sigma",
    data=df_reg_poly,
).fit(cov_type="HC3")

poly_model_2 = smf.ols(
    "mispricing ~ tau_days + sigma + C(bet_type)",
    data=df_reg_poly,
).fit(cov_type="HC3")

poly_model_3 = smf.ols(
    "abs_mispricing ~ prob_deribit + tau_days "
    "+ sigma + C(bet_type)",
    data=df_reg_poly,
).fit(cov_type="HC3")

poly_regression_results = pd.concat(
    [
        regression_to_table(
            poly_model_1,
            "Mispricing: controls",
        ),
        regression_to_table(
            poly_model_2,
            "Mispricing: controls + bet type FE",
        ),
        regression_to_table(
            poly_model_3,
            "Absolute mispricing: controls + bet type FE",
        ),
    ],
    ignore_index=True,
)

poly_regression_path = (
    TABLES_DIR / "regression_polymarket_terminal.csv"
)

poly_regression_results.to_csv(
    poly_regression_path,
    index=False,
)

display(
    poly_regression_results.style.format(
        precision=6,
        formatter={"p_value": "{:.3e}"},
    )
)

print(
    "Saved Polymarket terminal regressions to: "
    f"{poly_regression_path}"
)

,model,variable,coef,std_error,t_stat,p_value,ci_95_low,ci_95_high,nobs,r_squared
0,Mispricing: controls,Intercept,-0.006872,0.001878,-3.658738,2.535e-04,-0.010553,-0.003190,19649,0.010731
1,Mispricing: controls,tau_days,0.001941,0.000162,12.003307,3.414e-33,0.001624,0.002258,19649,0.010731
2,Mispricing: controls,sigma,0.016356,0.003589,4.557465,5.177e-06,0.009322,0.023390,19649,0.010731
3,Mispricing: controls + bet type FE,Intercept,-0.006737,0.001935,-3.480946,4.996e-04,-0.010530,-0.002944,19649,0.010741
4,Mispricing: controls + bet type FE,C(bet_type)[T.range],-0.000254,0.000574,-0.443515,6.574e-01,-0.001378,0.000870,19649,0.010741
5,Mispricing: controls + bet type FE,tau_days,0.001941,0.000162,11.989150,4.050e-33,0.001624,0.002258,19649,0.010741
6,Mispricing: controls + bet type FE,sigma,0.016320,0.003597,4.537007,5.706e-06,0.009270,0.023371,19649,0.010741
7,Absolute mispricing: controls + bet type FE,Intercept,0.028243,0.001722,16.399985,1.913e-60,0.024867,0.031618,19649,0.020182
8,Absolute mispricing: controls + bet type FE,C(bet_type)[T.range],0.001171,0.000585,2.001462,4.534e-02,0.000024,0.002317,19649,0.020182
9,Absolute mispricing: controls + bet type FE,prob_deribit,0.010870,0.000801,13.563518,6.590e-42,0.009299,0.012441,19649,0.020182


Saved Polymarket terminal regressions to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/regression_polymarket_terminal.csv


In [12]:
# ============================================================
# Polymarket regressions with clustered standard errors
# ============================================================

poly_cluster_model_1 = smf.ols(
    "mispricing ~ tau_days + sigma",
    data=df_reg_poly,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": df_reg_poly["market_id"]
    },
)

poly_cluster_model_2 = smf.ols(
    "mispricing ~ tau_days + sigma + C(bet_type)",
    data=df_reg_poly,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": df_reg_poly["market_id"]
    },
)

poly_cluster_model_3 = smf.ols(
    "abs_mispricing ~ prob_deribit + tau_days "
    "+ sigma + C(bet_type)",
    data=df_reg_poly,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": df_reg_poly["market_id"]
    },
)

poly_cluster_regression_results = pd.concat(
    [
        regression_to_table(
            poly_cluster_model_1,
            "Mispricing: controls, clustered by market",
        ),
        regression_to_table(
            poly_cluster_model_2,
            "Mispricing: controls + bet type FE, "
            "clustered by market",
        ),
        regression_to_table(
            poly_cluster_model_3,
            "Absolute mispricing: controls + bet type FE, "
            "clustered by market",
        ),
    ],
    ignore_index=True,
)

poly_cluster_regression_path = (
    TABLES_DIR / "regression_polymarket_terminal_clustered.csv"
)

poly_cluster_regression_results.to_csv(
    poly_cluster_regression_path,
    index=False,
)

display(
    poly_cluster_regression_results.style.format(
        precision=6,
        formatter={"p_value": "{:.3e}"},
    )
)

print(
    "Saved clustered Polymarket regressions to: "
    f"{poly_cluster_regression_path}"
)

,model,variable,coef,std_error,t_stat,p_value,ci_95_low,ci_95_high,nobs,r_squared
0,"Mispricing: controls, clustered by market",Intercept,-0.006872,0.002724,-2.522400,1.166e-02,-0.012211,-0.001532,19649,0.010731
1,"Mispricing: controls, clustered by market",tau_days,0.001941,0.000185,10.521357,6.887e-26,0.001580,0.002303,19649,0.010731
2,"Mispricing: controls, clustered by market",sigma,0.016356,0.005485,2.981952,2.864e-03,0.005606,0.027106,19649,0.010731
3,"Mispricing: controls + bet type FE, clustered by market",Intercept,-0.006737,0.002769,-2.432581,1.499e-02,-0.012165,-0.001309,19649,0.010741
4,"Mispricing: controls + bet type FE, clustered by market",C(bet_type)[T.range],-0.000254,0.000804,-0.316475,7.516e-01,-0.001830,0.001321,19649,0.010741
5,"Mispricing: controls + bet type FE, clustered by market",tau_days,0.001941,0.000185,10.513778,7.464e-26,0.001579,0.002303,19649,0.010741
6,"Mispricing: controls + bet type FE, clustered by market",sigma,0.016320,0.005485,2.975379,2.926e-03,0.005570,0.027071,19649,0.010741
7,"Absolute mispricing: controls + bet type FE, clustered by market",Intercept,0.028243,0.002254,12.530663,5.074e-36,0.023825,0.032660,19649,0.020182
8,"Absolute mispricing: controls + bet type FE, clustered by market",C(bet_type)[T.range],0.001171,0.000730,1.602497,1.090e-01,-0.000261,0.002602,19649,0.020182
9,"Absolute mispricing: controls + bet type FE, clustered by market",prob_deribit,0.010870,0.001072,10.143078,3.557e-24,0.008770,0.012970,19649,0.020182


Saved clustered Polymarket regressions to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/regression_polymarket_terminal_clustered.csv


In [13]:
# ============================================================
# Unified market regressions
# ============================================================

df_reg_unified = df_unified.copy()

if not df_reg_unified["market"].isin(
    ["kalshi", "polymarket"]
).all():
    raise ValueError("Missing or unexpected market labels.")

# Kalshi is the reference category.
df_reg_unified["market"] = pd.Categorical(
    df_reg_unified["market"],
    categories=["kalshi", "polymarket"],
)

df_reg_unified["contract_type"] = (
    df_reg_unified["contract_type"].astype("category")
)

for column in ["path_dependent_sample", "terminal_sample"]:
    if not pd.api.types.is_bool_dtype(df_reg_unified[column]):
        raise TypeError(f"{column} must contain boolean values.")
    if df_reg_unified[column].isna().any():
        raise ValueError(f"{column} contains missing values.")

df_reg_unified["path_dependent_int"] = (
    df_reg_unified["path_dependent_sample"].astype(int)
)

df_reg_unified_terminal = df_reg_unified.loc[
    df_reg_unified["terminal_sample"]
].copy()

assert "inference_cluster" in df_reg_unified.columns
assert df_reg_unified["inference_cluster"].notna().all()

# Signed mispricing: benchmark probability excluded.
unified_model_1 = smf.ols(
    "mispricing ~ C(market) + tau_days + sigma",
    data=df_reg_unified,
    missing="raise",
).fit(cov_type="HC3")

unified_model_2 = smf.ols(
    "mispricing ~ C(market) + path_dependent_int "
    "+ tau_days + sigma",
    data=df_reg_unified,
    missing="raise",
).fit(cov_type="HC3")

unified_model_3 = smf.ols(
    "mispricing ~ C(market) + tau_days + sigma",
    data=df_reg_unified_terminal,
    missing="raise",
).fit(cov_type="HC3")

# Supplementary absolute-mispricing specification.
unified_model_4 = smf.ols(
    "abs_mispricing ~ C(market) + path_dependent_int "
    "+ prob_deribit + tau_days + sigma",
    data=df_reg_unified,
    missing="raise",
).fit(cov_type="HC3")

unified_regression_results = pd.concat(
    [
        regression_to_table(
            unified_model_1,
            "Unified: platform indicator, HC3",
        ),
        regression_to_table(
            unified_model_2,
            "Unified: platform indicator + path-dependent flag, HC3",
        ),
        regression_to_table(
            unified_model_3,
            "Unified terminal only: platform indicator, HC3",
        ),
        regression_to_table(
            unified_model_4,
            "Unified absolute mispricing: platform indicator "
            "+ path-dependent flag, HC3",
        ),
    ],
    ignore_index=True,
)

unified_regression_path = (
    TABLES_DIR / "regression_unified_market.csv"
)

# Preserve full precision in the exported results.
unified_regression_results.to_csv(
    unified_regression_path,
    index=False,
)

display(
    unified_regression_results.style.format(
        precision=6,
        formatter={"p_value": "{:.3e}"},
    )
)
print(f"Saved unified regressions to: {unified_regression_path}")

,model,variable,coef,std_error,t_stat,p_value,ci_95_low,ci_95_high,nobs,r_squared
0,"Unified: platform indicator, HC3",Intercept,0.066497,0.004310,15.427802,1.064e-53,0.058049,0.074945,33107,0.281282
1,"Unified: platform indicator, HC3",C(market)[T.polymarket],-0.049285,0.002676,-18.419011,9.248e-76,-0.054530,-0.044041,33107,0.281282
2,"Unified: platform indicator, HC3",tau_days,0.000950,0.000014,68.471596,0.000e+00,0.000923,0.000977,33107,0.281282
3,"Unified: platform indicator, HC3",sigma,-0.007546,0.007884,-0.957083,3.385e-01,-0.022999,0.007907,33107,0.281282
4,"Unified: platform indicator + path-dependent flag, HC3",Intercept,0.068951,0.004256,16.202568,4.836e-59,0.060610,0.077291,33107,0.299956
5,"Unified: platform indicator + path-dependent flag, HC3",C(market)[T.polymarket],-0.058793,0.002647,-22.210753,2.704e-109,-0.063981,-0.053605,33107,0.299956
6,"Unified: platform indicator + path-dependent flag, HC3",path_dependent_int,0.034182,0.001318,25.941183,2.286e-148,0.031599,0.036764,33107,0.299956
7,"Unified: platform indicator + path-dependent flag, HC3",tau_days,0.000841,0.000015,55.337447,0.000e+00,0.000811,0.000871,33107,0.299956
8,"Unified: platform indicator + path-dependent flag, HC3",sigma,-0.012892,0.007742,-1.665244,9.586e-02,-0.028065,0.002282,33107,0.299956
9,"Unified terminal only: platform indicator, HC3",Intercept,0.054146,0.003292,16.448313,8.624e-61,0.047694,0.060598,22784,0.080555


Saved unified regressions to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/regression_unified_market.csv


In [14]:
# ============================================================
# Unified regressions with clustered standard errors
# ============================================================

# Polymarket observations are clustered by market.
# Kalshi strike bins are clustered by daily event.

assert "inference_cluster" in df_reg_unified.columns
assert (
    "inference_cluster"
    in df_reg_unified_terminal.columns
)

unified_cluster_model_1 = smf.ols(
    "mispricing ~ C(market) + tau_days + sigma",
    data=df_reg_unified,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
            df_reg_unified["inference_cluster"]
    },
)

unified_cluster_model_2 = smf.ols(
    "mispricing ~ C(market) + path_dependent_int "
    "+ tau_days + sigma",
    data=df_reg_unified,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
            df_reg_unified["inference_cluster"]
    },
)

unified_cluster_model_3 = smf.ols(
    "mispricing ~ C(market) + tau_days + sigma",
    data=df_reg_unified_terminal,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
            df_reg_unified_terminal[
                "inference_cluster"
            ]
    },
)

unified_cluster_model_4 = smf.ols(
    "abs_mispricing ~ C(market) "
    "+ path_dependent_int + prob_deribit "
    "+ tau_days + sigma",
    data=df_reg_unified,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
            df_reg_unified["inference_cluster"]
    },
)

unified_cluster_regression_results = pd.concat(
    [
        regression_to_table(
            unified_cluster_model_1,
            "Unified: platform indicator, "
            "clustered by empirical unit",
        ),
        regression_to_table(
            unified_cluster_model_2,
            "Unified: platform indicator + path-dependent flag, "
            "clustered by empirical unit",
        ),
        regression_to_table(
            unified_cluster_model_3,
            "Unified terminal only: platform indicator, "
            "clustered by empirical unit",
        ),
        regression_to_table(
            unified_cluster_model_4,
            "Unified absolute mispricing: platform indicator "
            "+ path-dependent flag, clustered by empirical unit",
        ),
    ],
    ignore_index=True,
)

unified_cluster_regression_path = (
    TABLES_DIR / "regression_unified_market_clustered.csv"
)

unified_cluster_regression_results.to_csv(
    unified_cluster_regression_path,
    index=False,
)

display(
    unified_cluster_regression_results.style.format(
        precision=6,
        formatter={"p_value": "{:.3e}"},
    )
)

print(
    "Saved clustered unified regressions to: "
    f"{unified_cluster_regression_path}"
)

,model,variable,coef,std_error,t_stat,p_value,ci_95_low,ci_95_high,nobs,r_squared
0,"Unified: platform indicator, clustered by empirical unit",Intercept,0.066497,0.016060,4.140445,3.466e-05,0.035019,0.097975,33107,0.281282
1,"Unified: platform indicator, clustered by empirical unit",C(market)[T.polymarket],-0.049285,0.004729,-10.422779,1.952e-25,-0.058553,-0.040017,33107,0.281282
2,"Unified: platform indicator, clustered by empirical unit",tau_days,0.000950,0.000159,5.974451,2.309e-09,0.000638,0.001262,33107,0.281282
3,"Unified: platform indicator, clustered by empirical unit",sigma,-0.007546,0.033993,-0.221988,8.243e-01,-0.074171,0.059079,33107,0.281282
4,"Unified: platform indicator + path-dependent flag, clustered by empirical unit",Intercept,0.068951,0.015728,4.383946,1.165e-05,0.038124,0.099777,33107,0.299956
5,"Unified: platform indicator + path-dependent flag, clustered by empirical unit",C(market)[T.polymarket],-0.058793,0.004350,-13.514844,1.278e-41,-0.067319,-0.050266,33107,0.299956
6,"Unified: platform indicator + path-dependent flag, clustered by empirical unit",path_dependent_int,0.034182,0.005395,6.335316,2.369e-10,0.023607,0.044756,33107,0.299956
7,"Unified: platform indicator + path-dependent flag, clustered by empirical unit",tau_days,0.000841,0.000163,5.170285,2.337e-07,0.000522,0.001160,33107,0.299956
8,"Unified: platform indicator + path-dependent flag, clustered by empirical unit",sigma,-0.012892,0.033244,-0.387791,6.982e-01,-0.078049,0.052265,33107,0.299956
9,"Unified terminal only: platform indicator, clustered by empirical unit",Intercept,0.054146,0.005156,10.501881,8.468e-26,0.044041,0.064251,22784,0.080555


Saved clustered unified regressions to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/regression_unified_market_clustered.csv


In [18]:
# ============================================================
# Probability alignment with the DVOL-based benchmark
# ============================================================

alignment_specs = [
    ("Polymarket terminal", df_poly_terminal, "market_id"),
    (KALSHI_MAIN_LABEL, df_kalshi, "event_ticker"),
]

alignment_models = {}
alignment_coefficient_tables = []
alignment_test_rows = []

for sample_name, df_sample, cluster_col in alignment_specs:
    required_cols = [
        "prob_market",
        "prob_deribit",
        cluster_col,
    ]
    df_alignment = df_sample[required_cols].copy()

    if df_alignment.isna().any().any():
        raise ValueError(
            f"{sample_name}: missing probabilities or cluster identifiers."
        )

    probabilities = df_alignment[
        ["prob_market", "prob_deribit"]
    ].to_numpy(dtype=float)

    if not np.isfinite(probabilities).all():
        raise ValueError(f"{sample_name}: non-finite probabilities.")

    if not ((probabilities >= 0) & (probabilities <= 1)).all():
        raise ValueError(f"{sample_name}: probabilities outside [0, 1].")

    if df_alignment["prob_deribit"].nunique() < 2:
        raise ValueError(f"{sample_name}: benchmark has no variation.")

    if df_alignment[cluster_col].nunique() < 2:
        raise ValueError(f"{sample_name}: fewer than two clusters.")

    model = smf.ols(
        "prob_market ~ prob_deribit",
        data=df_alignment,
        missing="raise",
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_alignment[cluster_col]},
        use_t=False,
    )

    alignment_models[sample_name] = model

    coefficients = regression_to_table(model, sample_name)
    coefficients["clusters"] = df_alignment[cluster_col].nunique()
    coefficients["cluster_variable"] = cluster_col
    alignment_coefficient_tables.append(coefficients)

    alpha_test = model.t_test("Intercept = 0")
    beta_test = model.t_test("prob_deribit = 1")
    joint_test = model.wald_test(
        "Intercept = 0, prob_deribit = 1",
        use_f=False,
        scalar=True,
    )

    alignment_test_rows.append({
        "sample": sample_name,
        "observations": int(model.nobs),
        "clusters": df_alignment[cluster_col].nunique(),
        "cluster_variable": cluster_col,
        "alpha": model.params["Intercept"],
        "beta": model.params["prob_deribit"],
        "alpha_zero_p_value": np.asarray(alpha_test.pvalue).item(),
        "beta_one_p_value": np.asarray(beta_test.pvalue).item(),
        "joint_chi2": np.asarray(joint_test.statistic).item(),
        "joint_df": 2,
        "joint_p_value": np.asarray(joint_test.pvalue).item(),
    })

probability_alignment_results = pd.concat(
    alignment_coefficient_tables,
    ignore_index=True,
)
probability_alignment_tests = pd.DataFrame(alignment_test_rows)

probability_alignment_path = (
    TABLES_DIR / "regression_probability_alignment_clustered.csv"
)
probability_alignment_tests_path = (
    TABLES_DIR / "probability_alignment_tests.csv"
)

probability_alignment_results.to_csv(
    probability_alignment_path, index=False
)
probability_alignment_tests.to_csv(
    probability_alignment_tests_path, index=False
)

display(
    probability_alignment_results.style.format(
        precision=6,
        formatter={"p_value": "{:.3e}"},
    )
)

display(
    probability_alignment_tests.style.format(
        precision=6,
        formatter={
            "alpha_zero_p_value": "{:.3e}",
            "beta_one_p_value": "{:.3e}",
            "joint_p_value": "{:.3e}",
        },
    )
)

print(f"Saved coefficients to: {probability_alignment_path}")
print(f"Saved hypothesis tests to: {probability_alignment_tests_path}")

,model,variable,coef,std_error,t_stat,p_value,ci_95_low,ci_95_high,nobs,r_squared,clusters,cluster_variable
0,Polymarket terminal,Intercept,0.003184,0.000458,6.959685,3.410e-12,0.002287,0.004081,19649,0.987246,2847,market_id
1,Polymarket terminal,prob_deribit,1.014286,0.001182,858.168055,0.000e+00,1.011969,1.016602,19649,0.987246,2847,market_id
2,"Kalshi timestamped, first trade within 30 minutes",Intercept,0.055880,0.013419,4.164248,3.124e-05,0.029579,0.082181,3135,0.239879,485,event_ticker
3,"Kalshi timestamped, first trade within 30 minutes",prob_deribit,1.059136,0.080108,13.221413,6.601e-40,0.902128,1.216144,3135,0.239879,485,event_ticker


,sample,observations,clusters,cluster_variable,alpha,beta,alpha_zero_p_value,beta_one_p_value,joint_chi2,joint_df,joint_p_value
0,Polymarket terminal,19649,2847,market_id,0.003184,1.014286,3.410e-12,1.236e-33,415.021465,2,7.572e-91
1,"Kalshi timestamped, first trade within 30 minutes",3135,485,event_ticker,0.055880,1.059136,3.124e-05,4.604e-01,716.774340,2,2.262e-156


Saved coefficients to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/regression_probability_alignment_clustered.csv
Saved hypothesis tests to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/probability_alignment_tests.csv


In [19]:
# ============================================================
# Robustness mispricing tests
# ============================================================

df_unified_terminal = (
    df_unified.loc[
        df_unified["terminal_sample"]
    ].copy()
)

robustness_samples = {
    "Main: Polymarket terminal":
        df_poly_terminal,
    "Polymarket terminal very-short":
        df_poly_terminal_very_short,
    "Polymarket path-dependent":
        df_poly_path_dep,
    KALSHI_MAIN_LABEL:
        df_kalshi,
    KALSHI_15M_LABEL:
        df_kalshi_15m,
    "Unified matched":
        df_unified,
    "Unified terminal only":
        df_unified_terminal,
}

robustness_rows = [
    mean_mispricing_test(
        df_sample,
        sample_name,
    )
    for sample_name, df_sample
    in robustness_samples.items()
]

robustness_results = pd.DataFrame(
    robustness_rows
)

robustness_results[round_cols] = (
    robustness_results[round_cols].round(6)
)

robustness_path = (
    TABLES_DIR
    / "robustness_mispricing_tests.csv"
)

robustness_results.to_csv(
    robustness_path,
    index=False,
)

display(robustness_results)

print(
    "Saved robustness tests to: "
    f"{robustness_path}"
)

,sample,observations,contracts,mean_mispricing,median_mispricing,sd_mispricing,std_error,t_stat,p_value,ci_95_low,ci_95_high
0,Main: Polymarket terminal,19649,2847,0.007551,0.002772,0.039818,0.000284,26.580929,0.0,0.006994,0.008107
1,Polymarket terminal very-short,19616,2847,0.007511,0.002740,0.039828,0.000284,26.413494,0.0,0.006954,0.008069
2,Polymarket path-dependent,10323,584,0.084643,0.030254,0.143066,0.001408,60.111877,0.0,0.081883,0.087404
3,"Kalshi timestamped, first trade within 30 minutes",3135,3135,0.063073,0.019875,0.147670,0.002637,23.915031,0.0,0.057902,0.068244
4,"Kalshi timestamped, first trade within 15 minutes",2554,2554,0.063855,0.026789,0.139220,0.002755,23.179580,0.0,0.058453,0.069257
5,Unified matched,33107,6566,0.036846,0.008653,0.103314,0.000568,64.892538,0.0,0.035733,0.037959
6,Unified terminal only,22784,5982,0.015190,0.003668,0.068795,0.000456,33.328920,0.0,0.014297,0.016084


Saved robustness tests to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/robustness_mispricing_tests.csv


In [21]:
# ============================================================
# Polymarket terminal results by time to maturity
# ============================================================

maturity_required = ["market_id", "tau_days", "mispricing"]
df_maturity = df_poly_terminal[maturity_required].copy()

if df_maturity.isna().any().any():
    raise ValueError("Missing values in the maturity analysis sample.")

if not np.isfinite(
    df_maturity[["tau_days", "mispricing"]].to_numpy(dtype=float)
).all():
    raise ValueError("Non-finite values in the maturity analysis sample.")

if not df_maturity["tau_days"].gt(0).all():
    raise ValueError("Maturity must be strictly positive.")

maturity_labels = [
    "0 < tau <= 1 day",
    "1 < tau <= 3 days",
    "tau > 3 days",
]

df_maturity["maturity_group"] = pd.cut(
    df_maturity["tau_days"],
    bins=[0, 1, 3, np.inf],
    labels=maturity_labels,
    right=True,
)

assert df_maturity["maturity_group"].notna().all()

maturity_rows = []

for label in maturity_labels:
    df_group = df_maturity.loc[
        df_maturity["maturity_group"].eq(label)
    ].copy()

    if df_group["market_id"].nunique() < 2:
        raise ValueError(f"{label}: fewer than two markets.")

    result = cluster_level_bootstrap_ci(
        df=df_group,
        sample_name=f"Polymarket terminal: {label}",
        cluster_col="market_id",
        n_boot=2000,
        random_seed=42,
    )

    result["maturity_group"] = label
    result["mean_tau_days"] = df_group["tau_days"].mean()
    maturity_rows.append(result)

maturity_group_results = pd.DataFrame(maturity_rows)

assert (
    maturity_group_results["observations"].sum()
    == len(df_poly_terminal)
)

maturity_group_path = (
    TABLES_DIR / "polymarket_terminal_maturity_groups.csv"
)
maturity_group_results.to_csv(maturity_group_path, index=False)

maturity_display_cols = [
    "maturity_group",
    "observations",
    "clusters",
    "mean_tau_days",
    "row_mean_mispricing",
    "cluster_mean_mispricing",
    "cluster_mean_ci_95_low",
    "cluster_mean_ci_95_high",
]

display(maturity_group_results[maturity_display_cols].round(6))
print(f"Saved maturity results to: {maturity_group_path}")

,maturity_group,observations,clusters,mean_tau_days,row_mean_mispricing,cluster_mean_mispricing,cluster_mean_ci_95_low,cluster_mean_ci_95_high
0,0 < tau <= 1 day,2806,2786,0.675421,0.002853,0.002883,0.001467,0.004410
1,1 < tau <= 3 days,5578,2802,2.175012,0.004473,0.004419,0.003239,0.005657
2,tau > 3 days,11265,2844,5.198211,0.010244,0.010178,0.009186,0.011169


Saved maturity results to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/polymarket_terminal_maturity_groups.csv


In [22]:
# ============================================================
# Monthly-level robustness tests
# ============================================================

def add_month_column(df, date_col):
    df_out = df.copy()

    if not pd.api.types.is_datetime64_any_dtype(
        df_out[date_col]
    ):
        df_out[date_col] = pd.to_datetime(
            df_out[date_col],
            utc=True,
            errors="coerce",
        )

    df_out["month"] = pd.to_datetime(
        df_out[date_col].dt.strftime("%Y-%m-01"),
        utc=True,
    )

    return df_out


def mean_test_series(x, sample_name):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n < 2:
        raise ValueError(
            f"{sample_name} has fewer than two periods."
        )

    mean_value = x.mean()
    median_value = x.median()
    sd_value = x.std(ddof=1)
    se_value = sd_value / np.sqrt(n)

    if se_value == 0:
        raise ValueError(
            f"{sample_name} has zero standard error."
        )

    t_stat = mean_value / se_value
    p_value = 2 * stats.t.sf(
        abs(t_stat),
        df=n - 1,
    )

    critical_value = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    return {
        "sample": sample_name,
        "months": n,
        "mean_mispricing": mean_value,
        "median_mispricing": median_value,
        "sd_mispricing": sd_value,
        "std_error": se_value,
        "t_stat": t_stat,
        "p_value": p_value,
        "ci_95_low": (
            mean_value
            - critical_value * se_value
        ),
        "ci_95_high": (
            mean_value
            + critical_value * se_value
        ),
    }


monthly_test_samples = {
    "Polymarket terminal":
        add_month_column(
            df_poly_terminal,
            "date",
        ),
    "Polymarket path-dependent":
        add_month_column(
            df_poly_path_dep,
            "date",
        ),
    KALSHI_MAIN_LABEL:
        add_month_column(
            df_kalshi,
            "observation_date",
        ),
    KALSHI_15M_LABEL:
        add_month_column(
            df_kalshi_15m,
            "observation_date",
        ),
    "Unified matched":
        add_month_column(
            df_unified,
            "date",
        ),
}

monthly_rows = []

for sample_name, df_sample in monthly_test_samples.items():
    monthly_average = (
        df_sample
        .groupby("month", as_index=False)
        .agg(
            monthly_mispricing=(
                "mispricing",
                "mean",
            ),
            observations=(
                "mispricing",
                "size",
            ),
        )
    )

    result = mean_test_series(
        monthly_average["monthly_mispricing"],
        sample_name,
    )

    result["avg_monthly_observations"] = (
        monthly_average["observations"].mean()
    )
    result["min_monthly_observations"] = (
        monthly_average["observations"].min()
    )
    result["max_monthly_observations"] = (
        monthly_average["observations"].max()
    )

    monthly_rows.append(result)

monthly_level_results = pd.DataFrame(
    monthly_rows
)

monthly_round_cols = [
    "mean_mispricing",
    "median_mispricing",
    "sd_mispricing",
    "std_error",
    "t_stat",
    "p_value",
    "ci_95_low",
    "ci_95_high",
    "avg_monthly_observations",
]

monthly_level_results[monthly_round_cols] = (
    monthly_level_results[
        monthly_round_cols
    ].round(6)
)

monthly_level_path = (
    TABLES_DIR
    / "monthly_level_mispricing_tests.csv"
)

monthly_level_results.to_csv(
    monthly_level_path,
    index=False,
)

display(monthly_level_results)

print(
    "Saved monthly-level tests to: "
    f"{monthly_level_path}"
)

,sample,months,mean_mispricing,median_mispricing,sd_mispricing,std_error,t_stat,p_value,ci_95_low,ci_95_high,avg_monthly_observations,min_monthly_observations,max_monthly_observations
0,Polymarket terminal,28,0.017142,0.009638,0.015608,0.002950,5.811543,0.000003,0.011090,0.023194,701.750000,20,4060
1,Polymarket path-dependent,27,0.109340,0.073358,0.102348,0.019697,5.551099,0.000008,0.068852,0.149827,382.333333,10,978
2,"Kalshi timestamped, first trade within 30 minutes",17,0.062546,0.062743,0.018193,0.004412,14.175248,0.000000,0.053192,0.071900,184.411765,42,410
3,"Kalshi timestamped, first trade within 15 minutes",17,0.066058,0.067941,0.020031,0.004858,13.597220,0.000000,0.055759,0.076357,150.235294,32,318
4,Unified matched,28,0.067155,0.060738,0.067194,0.012698,5.288462,0.000014,0.041100,0.093210,1182.392857,20,5109


Saved monthly-level tests to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/monthly_level_mispricing_tests.csv


In [23]:
# ============================================================
# Filtered monthly-level robustness tests
# ============================================================

MIN_MONTHLY_OBS = 100

filtered_monthly_rows = []

for sample_name, df_sample in monthly_test_samples.items():
    monthly_average = (
        df_sample
        .groupby("month", as_index=False)
        .agg(
            monthly_mispricing=(
                "mispricing",
                "mean",
            ),
            observations=(
                "mispricing",
                "size",
            ),
        )
    )

    monthly_average = monthly_average.loc[
        monthly_average["observations"]
        >= MIN_MONTHLY_OBS
    ].copy()

    if len(monthly_average) < 2:
        print(
            f"Skipping {sample_name}: fewer than two "
            f"months satisfy the filter."
        )
        continue

    result = mean_test_series(
        monthly_average["monthly_mispricing"],
        sample_name,
    )

    result["min_monthly_obs_filter"] = (
        MIN_MONTHLY_OBS
    )
    result["avg_monthly_observations"] = (
        monthly_average["observations"].mean()
    )
    result["min_monthly_observations"] = (
        monthly_average["observations"].min()
    )
    result["max_monthly_observations"] = (
        monthly_average["observations"].max()
    )

    filtered_monthly_rows.append(result)

filtered_monthly_results = pd.DataFrame(
    filtered_monthly_rows
)

filtered_monthly_results[
    monthly_round_cols
] = filtered_monthly_results[
    monthly_round_cols
].round(6)

filtered_monthly_path = (
    TABLES_DIR
    / "monthly_level_mispricing_tests_filtered.csv"
)

filtered_monthly_results.to_csv(
    filtered_monthly_path,
    index=False,
)

display(filtered_monthly_results)

print(
    "Saved filtered monthly-level tests to: "
    f"{filtered_monthly_path}"
)

,sample,months,mean_mispricing,median_mispricing,sd_mispricing,std_error,t_stat,p_value,ci_95_low,ci_95_high,min_monthly_obs_filter,avg_monthly_observations,min_monthly_observations,max_monthly_observations
0,Polymarket terminal,16,0.006445,0.007526,0.004115,0.001029,6.264970,0.000015,0.004252,0.008637,100,1202.437500,120,4060
1,Polymarket path-dependent,25,0.105288,0.073358,0.095380,0.019076,5.519380,0.000011,0.065917,0.144659,100,411.240000,132,978
2,"Kalshi timestamped, first trade within 30 minutes",16,0.063910,0.063473,0.017868,0.004467,14.306957,0.000000,0.054389,0.073432,100,193.312500,110,410
3,"Kalshi timestamped, first trade within 15 minutes",14,0.062783,0.067397,0.015222,0.004068,15.432493,0.000000,0.053994,0.071572,100,167.071429,105,318
4,Unified matched,26,0.065722,0.055209,0.069203,0.013572,4.842528,0.000056,0.037770,0.093673,100,1271.000000,163,5109


Saved filtered monthly-level tests to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/monthly_level_mispricing_tests_filtered.csv


In [24]:
# ============================================================
# Save main empirical findings summary
# ============================================================

def get_cluster_result(sample_name):
    selected = cluster_bootstrap_results.loc[
        cluster_bootstrap_results["sample"]
        == sample_name
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Expected one cluster result for {sample_name}, "
            f"found {len(selected)}."
        )

    return selected.iloc[0]


def get_filtered_monthly_result(sample_name):
    selected = filtered_monthly_results.loc[
        filtered_monthly_results["sample"]
        == sample_name
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Expected one monthly result for {sample_name}, "
            f"found {len(selected)}."
        )

    return selected.iloc[0]


main_cluster = get_cluster_result(
    "Polymarket terminal"
)
very_short_cluster = get_cluster_result(
    "Polymarket terminal very-short-horizon"
)
path_cluster = get_cluster_result(
    "Polymarket path-dependent"
)
kalshi_30m_cluster = get_cluster_result(
    KALSHI_MAIN_LABEL
)
kalshi_15m_cluster = get_cluster_result(
    KALSHI_15M_LABEL
)
monthly_main = get_filtered_monthly_result(
    "Polymarket terminal"
)

main_findings = pd.DataFrame([
    {
        "finding": "Main result",
        "sample": "Polymarket terminal",
        "mean_mispricing":
            main_cluster["cluster_mean_mispricing"],
        "ci_95_low":
            main_cluster["cluster_mean_ci_95_low"],
        "ci_95_high":
            main_cluster["cluster_mean_ci_95_high"],
        "inference_type":
            "Market-level bootstrap",
    },
    {
        "finding":
            "Supplementary: exclusion of maturities above 7 days",
        "sample":
            "Polymarket terminal very-short-horizon",
        "mean_mispricing":
            very_short_cluster[
                "cluster_mean_mispricing"
            ],
        "ci_95_low":
            very_short_cluster[
                "cluster_mean_ci_95_low"
            ],
        "ci_95_high":
            very_short_cluster[
                "cluster_mean_ci_95_high"
            ],
        "inference_type":
            "Market-level bootstrap",
    },
    {
        "finding":
            "Path-dependent comparison",
        "sample":
            "Polymarket path-dependent",
        "mean_mispricing":
            path_cluster["cluster_mean_mispricing"],
        "ci_95_low":
            path_cluster["cluster_mean_ci_95_low"],
        "ci_95_high":
            path_cluster["cluster_mean_ci_95_high"],
        "inference_type":
            "Market-level bootstrap",
    },
    {
        "finding":
            "Kalshi supplementary comparison",
        "sample":
            KALSHI_MAIN_LABEL,
        "mean_mispricing":
            kalshi_30m_cluster[
                "cluster_mean_mispricing"
            ],
        "ci_95_low":
            kalshi_30m_cluster[
                "cluster_mean_ci_95_low"
            ],
        "ci_95_high":
            kalshi_30m_cluster[
                "cluster_mean_ci_95_high"
            ],
        "inference_type":
            "Event-level bootstrap",
    },
    {
        "finding":
            "Kalshi 15-minute robustness",
        "sample":
            KALSHI_15M_LABEL,
        "mean_mispricing":
            kalshi_15m_cluster[
                "cluster_mean_mispricing"
            ],
        "ci_95_low":
            kalshi_15m_cluster[
                "cluster_mean_ci_95_low"
            ],
        "ci_95_high":
            kalshi_15m_cluster[
                "cluster_mean_ci_95_high"
            ],
        "inference_type":
            "Event-level bootstrap",
    },
    {
        "finding":
            "Filtered monthly robustness",
        "sample":
            "Polymarket terminal, monthly filtered",
        "mean_mispricing":
            monthly_main["mean_mispricing"],
        "ci_95_low":
            monthly_main["ci_95_low"],
        "ci_95_high":
            monthly_main["ci_95_high"],
        "inference_type":
            "Filtered monthly t-test",
    },
])

# Add the main maturity robustness results.
maturity_findings = pd.DataFrame({
    "finding": "Maturity-group robustness",
    "sample": maturity_group_results["sample"],
    "mean_mispricing": (
        maturity_group_results["cluster_mean_mispricing"]
    ),
    "ci_95_low": (
        maturity_group_results["cluster_mean_ci_95_low"]
    ),
    "ci_95_high": (
        maturity_group_results["cluster_mean_ci_95_high"]
    ),
    "inference_type": "Market-level bootstrap within maturity group",
})

main_findings = pd.concat(
    [main_findings, maturity_findings],
    ignore_index=True,
)

numeric_findings_cols = [
    "mean_mispricing",
    "ci_95_low",
    "ci_95_high",
]

main_findings[numeric_findings_cols] = (
    main_findings[numeric_findings_cols].round(6)
)

main_findings_path = (
    TABLES_DIR / "main_empirical_findings.csv"
)

main_findings.to_csv(
    main_findings_path,
    index=False,
)

display(main_findings)

print(
    "Saved main empirical findings to: "
    f"{main_findings_path}"
)

,finding,sample,mean_mispricing,ci_95_low,ci_95_high,inference_type
0,Main result,Polymarket terminal,0.007496,0.006709,0.008285,Market-level bootstrap
1,Supplementary: exclusion of maturities above 7...,Polymarket terminal very-short-horizon,0.007483,0.006704,0.008271,Market-level bootstrap
2,Path-dependent comparison,Polymarket path-dependent,0.076566,0.067337,0.085983,Market-level bootstrap
3,Kalshi supplementary comparison,"Kalshi timestamped, first trade within 30 minutes",0.070670,0.064565,0.077081,Event-level bootstrap
4,Kalshi 15-minute robustness,"Kalshi timestamped, first trade within 15 minutes",0.076731,0.070557,0.083126,Event-level bootstrap
5,Filtered monthly robustness,"Polymarket terminal, monthly filtered",0.006445,0.004252,0.008637,Filtered monthly t-test
6,Maturity-group robustness,Polymarket terminal: 0 < tau <= 1 day,0.002883,0.001467,0.004410,Market-level bootstrap within maturity group
7,Maturity-group robustness,Polymarket terminal: 1 < tau <= 3 days,0.004419,0.003239,0.005657,Market-level bootstrap within maturity group
8,Maturity-group robustness,Polymarket terminal: tau > 3 days,0.010178,0.009186,0.011169,Market-level bootstrap within maturity group


Saved main empirical findings to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/main_empirical_findings.csv


In [25]:
# ============================================================
# Final empirical analysis export summary
# ============================================================

print("=" * 70)
print("Final empirical analysis export summary")
print("=" * 70)

print("Input datasets:")
print(f"  Polymarket terminal:    {poly_terminal_path}")
print(f"  Polymarket short:       {poly_terminal_short_path}")
print(f"  Polymarket very short:  {poly_terminal_very_short_path}")
print(f"  Polymarket path:        {poly_path_dep_path}")
print(f"  Kalshi 30 minutes:      {kalshi_matched_path}")
print(f"  Kalshi 15 minutes:      {kalshi_matched_15m_path}")
print(f"  Unified:                {unified_path}")

print("\nSaved empirical tables:")
print(f"  Mean tests:                 {mean_tests_path}")
print(f"  Row bootstrap:              {row_bootstrap_path}")
print(f"  Cluster bootstrap:          {cluster_bootstrap_path}")
print(f"  Polymarket regressions:     {poly_regression_path}")
print(f"  Polymarket clustered:       {poly_cluster_regression_path}")
print(f"  Unified regressions:        {unified_regression_path}")
print(f"  Unified clustered:          {unified_cluster_regression_path}")
print(f"  Robustness tests:           {robustness_path}")
print(f"  Monthly tests:              {monthly_level_path}")
print(f"  Filtered monthly tests:     {filtered_monthly_path}")
print(f"  Main findings:              {main_findings_path}")

main_row = mean_mispricing_tests.loc[
    mean_mispricing_tests["sample"]
    == "Polymarket terminal"
].iloc[0]

main_cluster_row = cluster_bootstrap_results.loc[
    cluster_bootstrap_results["sample"]
    == "Polymarket terminal"
].iloc[0]

kalshi_row = mean_mispricing_tests.loc[
    mean_mispricing_tests["sample"]
    == KALSHI_MAIN_LABEL
].iloc[0]

kalshi_cluster_row = cluster_bootstrap_results.loc[
    cluster_bootstrap_results["sample"]
    == KALSHI_MAIN_LABEL
].iloc[0]

kalshi_15m_cluster_row = cluster_bootstrap_results.loc[
    cluster_bootstrap_results["sample"]
    == KALSHI_15M_LABEL
].iloc[0]

monthly_row = monthly_level_results.loc[
    monthly_level_results["sample"]
    == "Polymarket terminal"
].iloc[0]

filtered_monthly_row = filtered_monthly_results.loc[
    filtered_monthly_results["sample"]
    == "Polymarket terminal"
].iloc[0]

print("\nMain Polymarket result:")
print(
    "  Row-level mean: "
    f"{main_row['mean_mispricing']:.4f} "
    f"(95% CI: {main_row['ci_95_low']:.4f}, "
    f"{main_row['ci_95_high']:.4f})"
)
print(
    "  Market-level bootstrap mean: "
    f"{main_cluster_row['cluster_mean_mispricing']:.4f} "
    f"(95% CI: "
    f"{main_cluster_row['cluster_mean_ci_95_low']:.4f}, "
    f"{main_cluster_row['cluster_mean_ci_95_high']:.4f})"
)

print("\nKalshi supplementary result:")
print(
    "  30-minute row-level mean: "
    f"{kalshi_row['mean_mispricing']:.4f}"
)
print(
    "  30-minute event-level mean: "
    f"{kalshi_cluster_row['cluster_mean_mispricing']:.4f} "
    f"(95% CI: "
    f"{kalshi_cluster_row['cluster_mean_ci_95_low']:.4f}, "
    f"{kalshi_cluster_row['cluster_mean_ci_95_high']:.4f})"
)
print(
    "  15-minute event-level mean: "
    f"{kalshi_15m_cluster_row['cluster_mean_mispricing']:.4f} "
    f"(95% CI: "
    f"{kalshi_15m_cluster_row['cluster_mean_ci_95_low']:.4f}, "
    f"{kalshi_15m_cluster_row['cluster_mean_ci_95_high']:.4f})"
)

print("\nMonthly robustness:")
print(
    "  All months: "
    f"{monthly_row['mean_mispricing']:.4f} "
    f"(p-value: {monthly_row['p_value']:.6g})"
)
print(
    "  Filtered months: "
    f"{filtered_monthly_row['mean_mispricing']:.4f} "
    f"(p-value: "
    f"{filtered_monthly_row['p_value']:.6g})"
)

required_output_paths = [
    mean_tests_path,
    row_bootstrap_path,
    cluster_bootstrap_path,
    poly_regression_path,
    poly_cluster_regression_path,
    unified_regression_path,
    unified_cluster_regression_path,
    robustness_path,
    monthly_level_path,
    filtered_monthly_path,
    main_findings_path,
]

required_output_paths.extend([
    probability_alignment_path,
    probability_alignment_tests_path,
    maturity_group_path,
])

print("\nAdditional empirical tables:")
print(f"  Probability alignment: {probability_alignment_path}")
print(f"  Alignment tests:       {probability_alignment_tests_path}")
print(f"  Maturity groups:       {maturity_group_path}")

missing_outputs = [
    path for path in required_output_paths
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        f"Missing empirical outputs: {missing_outputs}"
    )

print("\nAll required empirical outputs exist.")
print("06_empirical_analysis.ipynb completed successfully.")

Final empirical analysis export summary
Input datasets:
  Polymarket terminal:    /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_terminal.csv
  Polymarket short:       /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_terminal_short.csv
  Polymarket very short:  /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_terminal_very_short.csv
  Polymarket path:        /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_deribit_matched_path_dependent.csv
  Kalshi 30 minutes:      /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_deribit_matched.csv
  Kalshi 15 minutes:      /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/kalshi_deribit_matched_15m.csv
  Unified:                /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/unified_prediction_market_de